# Q2 — Named Entity Recognition (CoNLL-2003)
**Models:** CRF (CPU) → BiLSTM-CRF (GPU) → BERT NER (GPU)

**Dataset:** Full CoNLL-2003

**Runtime:** T4 GPU required

## 1. Environment Setup

In [ ]:
import os, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/467-takehome-outputs/q2'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print('Drive mounted, output dir ready.')

In [ ]:
REPO_DIR = '/content/467-takehome'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/zubeyralmaho/467-takehome.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Run CRF Baseline (CPU)

In [ ]:
!python -m src.q2_ner.main \
    --config configs/q2.yaml \
    --final-eval \
    --override \
        models.crf.enabled=true \
        models.bilstm_crf.enabled=false \
        models.bert.enabled=false

In [ ]:
import glob, shutil
latest_run = sorted(glob.glob('outputs/q2/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_crf')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'CRF results saved to Drive: {dest}')

## 3. Run BiLSTM-CRF (GPU)

Tuned hyperparams for better F1 (target: 0.85+).

In [ ]:
!python -m src.q2_ner.main \
    --config configs/q2.yaml \
    --final-eval \
    --override \
        models.crf.enabled=false \
        models.bilstm_crf.enabled=true \
        models.bert.enabled=false \
        models.bilstm_crf.hidden_dim=256 \
        models.bilstm_crf.embedding_dim=128 \
        models.bilstm_crf.num_layers=2 \
        models.bilstm_crf.dropout=0.5 \
        models.bilstm_crf.learning_rate=0.0015 \
        models.bilstm_crf.max_epochs=15 \
        models.bilstm_crf.early_stopping_patience=4 \
        models.bilstm_crf.batch_size=32

In [ ]:
latest_run = sorted(glob.glob('outputs/q2/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_bilstm_crf')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'BiLSTM-CRF results saved to Drive: {dest}')

## 4. Run BERT NER (GPU)

In [ ]:
!python -m src.q2_ner.main \
    --config configs/q2.yaml \
    --final-eval \
    --override \
        models.crf.enabled=false \
        models.bilstm_crf.enabled=false \
        models.bert.enabled=true \
        models.bert.max_epochs=5 \
        models.bert.batch_size=16 \
        models.bert.learning_rate=5e-5

In [ ]:
latest_run = sorted(glob.glob('outputs/q2/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_bert')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'BERT NER results saved to Drive: {dest}')

## 5. Results Summary

In [ ]:
import json

print('=== Q2 Results Summary ===')
for run_dir in sorted(glob.glob('outputs/q2/run_*')):
    metrics_file = os.path.join(run_dir, 'metrics.json')
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            metrics = json.load(f)
        print(f'\n--- {os.path.basename(run_dir)} ---')
        print(json.dumps(metrics, indent=2, default=str)[:1500])